# Facility finder - local EDA
Profiles the three exported tables. Every field is a *claim to verify*, not ground truth.

In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
D = "."
def load(name): return pd.read_csv(f"{D}/{name}.csv", dtype=str, keep_default_na=False, na_values=[""])
fac  = load("facilities")
pin  = load("india_post_pincode_directory")
nfhs = load("nfhs_5_district_health_indicators")
for n, d in [("facilities", fac), ("pincode", pin), ("nfhs", nfhs)]:
    print(f"{n:12s} {d.shape[0]:>7,} rows x {d.shape[1]} cols")

## 1. Facilities: schema and coverage
Compare against the PDF: description 100%, capability 99.7%, procedure 92.5%, equipment 77%, year 47.8%, doctors 36.4%, capacity 25.2%.

In [ ]:
cov = (fac.notna() & (fac.apply(lambda s: s.str.strip() != "" if s.dtype==object else True))).mean().sort_values()
display_df = pd.DataFrame({"coverage": cov.round(3), "n_unique": fac.nunique(), "example": fac.apply(lambda s: s.dropna().iloc[0][:80] if s.notna().any() else None)})
display_df

In [ ]:
fac.head(3).T

## 2. Duplicates and merged records

In [ ]:
print("exact duplicate rows:", fac.duplicated().sum())
name_col = next((c for c in fac.columns if c.lower() in ("name","facility_name")), fac.columns[0])
print("name column:", name_col)
dups = fac[fac.duplicated(name_col, keep=False)].sort_values(name_col)
print("rows sharing a name:", len(dups))
dups.head(20)

## 3. Location sanity
Coordinates should be inside India and plausible for the stated state.

In [ ]:
lat = next((c for c in fac.columns if "lat" in c.lower()), None)
lon = next((c for c in fac.columns if c.lower().startswith("lon") or "lng" in c.lower()), None)
print("lat/lon columns:", lat, lon)
if lat and lon:
    g = fac.assign(_lat=pd.to_numeric(fac[lat], errors="coerce"), _lon=pd.to_numeric(fac[lon], errors="coerce"))
    print("missing coords:", g[["_lat","_lon"]].isna().any(axis=1).sum())
    out = g[(g._lat<6)|(g._lat>37.5)|(g._lon<68)|(g._lon>98)]
    print("outside India bbox:", len(out))
    state = next((c for c in fac.columns if "state" in c.lower()), None)
    if state:
        med = g.groupby(state)[["_lat","_lon"]].transform("median")
        dist = np.hypot(g._lat-med._lat, g._lon-med._lon)
        print("far (>8 deg) from own-state median:", (dist>8).sum())

## 4. Text fields

In [ ]:
text_cols = [c for c in fac.columns if any(k in c.lower() for k in ("description","capab","procedure","equip","special"))]
print(text_cols)
for c in text_cols:
    s = fac[c].dropna()
    print(f"{c}: median length {s.str.len().median():.0f}, empty-like (<=2 chars) {(s.str.len()<=2).sum()}")
fac[text_cols].head(3).T

## 5. Service evidence prototype
Search every text column for a service's keywords and keep the matching snippet as a citation.

In [ ]:
import re
SERVICES = {
    "dialysis": r"dialysis|haemodialysis|hemodialysis|renal replacement|nephrology",
    "nicu": r"nicu|neonatal intensive|newborn intensive",
    "c-section": r"c-?section|caesarean|cesarean|obstetric",
    "cardiac surgery": r"cardiac surgery|cardiothoracic|cabg|bypass surgery|open.heart",
    "mri": r"mri|magnetic resonance",
    "burn care": r"burn (unit|care|ward|centre|center)|plastic surgery",
}
def evidence(service, cols=text_cols, width=80):
    pat = re.compile(SERVICES[service], re.I); rows = []
    for i, r in fac[cols].iterrows():
        hits = []
        for c in cols:
            v = r[c]
            if isinstance(v, str):
                m = pat.search(v)
                if m: hits.append((c, v[max(0,m.start()-width):m.end()+width].replace("
"," ")))
        if hits: rows.append((i, len(hits), hits))
    return rows
for s in SERVICES:
    print(f"{s:16s} facilities with any text match: {len(evidence(s)):,}")

## 6. Reference tables

In [ ]:
pin.head(3).T

In [ ]:
nfhs.head(3).T